In [ ]:
import ssl
print(ssl.get_default_verify_paths())

In [ ]:
"""
Author: Margaret Woon
Last Updated Date: 22/08/2022
Version: 0.1
Description: To scrape a specific confluence pages data dictionary
Re-written from Adam's scrap_confluence.py 
"""
import os
from typing import Any, Dict, List

from bs4 import BeautifulSoup
import pandas as pd
import requests
from requests.auth import HTTPBasicAuth
from datetime import datetime
import json

def create_df(table_in):

        n_columns = 0
        n_rows=0
        column_names = []


        # Find number of rows and columns
        # we also find the column titles if we can
        for row in table_in.find_all('tr'):

            # Determine the number of rows in the table
            td_tags = row.find_all('td')
            if len(td_tags) > 0:
                n_rows+=1
                if n_columns == 0:
                    # Set the number of columns for our table
                    n_columns = len(td_tags)

            # Handle column names if we find them
            th_tags = row.find_all('th') 
            if len(th_tags) > 0 and len(column_names) == 0:
                for th in th_tags:
                    column_names.append(th.get_text())

        # Safeguard on Column Titles
        if len(column_names) > 0 and len(column_names) != n_columns:
            raise Exception("Column titles do not match the number of columns")

        columns = column_names if len(column_names) > 0 else range(0,n_columns)
        df = pd.DataFrame(columns = columns,
                          index= range(0,n_rows))
        row_marker = 0
        for row in table_in.find_all('tr'):
            column_marker = 0
            columns = row.find_all('td')
            for column in columns:
                df.iat[row_marker,column_marker] = column.get_text()
                #print("columns text nnnnnnnnnnnnnnnnn ",column.get_text() )
                column_marker += 1
            if len(columns) > 0:
                row_marker += 1

        # Convert to float if possible
        for col in df:
            try:
                df[col] = df[col].astype(float)
            except ValueError:
                pass

        return df
    
    

def get_paginated_pages(
    base_url: str, init_url: str, auth: HTTPBasicAuth, proxies: Dict[str, Any]
) -> List[Dict[str, Any]]:
    """
    Name:
        get_paginated_pages

    Inputs:
        base_url: str -> Base of URL to request from
        init_url: str -> Initial suffix of URL to request from
        auth: str -> HTTPBasicAuth object to authenticate API request

    Outputs:
        pages: List[Dict[str, Any]] -> A list of page results from API

    Description:
        This function is used to retrieve a list of information about
        confluence pages based on a paginated API (i.e. requiring multiple
        requests based on default limit of 25 items per page).
    """
    url = base_url + init_url
    pages = []

    print("url {0}".format(url))
    response = requests.get(url, auth=auth, proxies=proxies)

    
    txt = ''
    
    txt = response.text.replace("<br/>","\n")


    ### Calling soup
    soup = BeautifulSoup(txt, 'lxml')

#     print("soup {0}".format(soup))
    
    cnt = 0
    table_cnt = 0
    for table in soup.find_all('table'):
        table_cnt += 1
        cnt += 1
#         print(f"table {cnt} &&&&&&&&&&&&&       ", table, "\n\n\n\n")

        ### Due to the setup in the template;\
        ### this is configurable in config file confluenceSpecDocConfig.txt
        ### confluence_table = 6 for Cadencie bronze, confluence_table = 5 for BanCS bronze
        if cnt == 6: 
#             print(f"table {cnt} @@@@@@@@@@@@@@       ", table, "\n\n\n\n")


            df = create_df(table)
#             print("df       000000 {0}".format(df))
            
            if os.path.exists(excel_filename):
                try :
                    with pd.ExcelWriter(os.getcwd()+"\\"+excel_filename, engine='openpyxl', mode='a') as writer:  
                        df.to_excel(writer, sheet_name=self.dataset_name, index = None)
                except :
                     print(f"Fail to append tab {dataset_name}")
                else :
                    print(f"{dataset_name} tab append succeeded.")
                        
            else:
                try :
                    with pd.ExcelWriter(os.getcwd()+"\\"+excel_filename) as writer:  
                        df.to_excel(writer, sheet_name=dataset_name, index = None)
                except :
                    print(f"Fail to append tab {dataset_name}")
                else :
                    print(f"{dataset_name} tab append succeeded.")
    
#     return pages


# Setup

dataset_name ='balance_interest_table' 

filename = 'Presto'

page_id ='2075721729'

now = datetime.now()
now_yyyymmdd_hhmmss = now.strftime("%Y%m%d %H:%M:%S")

now_yyyymmdd = now.strftime("%Y%m%d")

print("Job proceed at {}".format(now_yyyymmdd_hhmmss))

excel_filename = filename+'_'+now_yyyymmdd+'.xlsx'

# EMAIL = os.getenv("CONFLUENCE_EMAIL")
# API_TOKEN = os.getenv("CONFLUENCE_API_TOKEN")

EMAIL = 'margaret.woon@gsb.com.au'

API_TOKEN=\
"ATATT3xFfGF0YXDfbV5dp-7xBSdIvNdFbD2ypNUIUbTHRYlR1Q31Ts5LyAMRcR8dmFu7NeKg6sL-8EAAkzc7HWxdVxpWfQdAoAObAz0cD285OjuRk4QgAmqUoxQLt69BJG7xeZJt4qAlmf63xLOrY0THJ5j5TsiUnWMendpT_HwZ5xNxsdFWL9I=85B81C61"



AUTH = HTTPBasicAuth(EMAIL, API_TOKEN)

print("PROXIES {0}".format(PROXIES))

PROXIES = {'http': 'http://165.225.114.22:443', 'https': 'http://165.225.114.22:443'}


BASE_URL = "https://cua.atlassian.net/wiki"
INIT_PARENT_URL='/spaces/DP/pages/2075721729/balance+interest+table'

parent_pages = get_paginated_pages(BASE_URL, INIT_PARENT_URL, auth=AUTH, proxies = PROXIES)


now = datetime.now()
now_yyyymmdd_hhmmss = now.strftime("%Y%m%d %H:%M:%S")
print("\n\nJob completed at {}".format(now_yyyymmdd_hhmmss))

# print("\nConfiguration url count: {}".format(len(df_url)))
# print("\nExtract count: {}".format(last_cnt))



